# Colorless FDN

FDN optimized for reduced metallic ringing (perceptually colorless reverberation). Original method published in *"Differentiable Feedback Delay Network for Colorless Reverberation," G Dal Santo, K Prawda, SJ Schlecht, V Välimäki, 26th International Conference on Digital Audio Effects (DAFx23), 244-251.*

Parameters are loaded from `.mat` files (e.g. from [diff-fdn-colorless](https://github.com/gdalsanto/diff-fdn-colorless)). The impulse response is computed with `pyFDN.dss2impz`. Modal decomposition (residue histogram) is omitted; pyFDN does not yet provide `dss2pr` (TODO).

- Original script in Matlab: Gloria Dal Santo, Wed, 18. Oct 2023
- Python translation: Sebastian J. Schlecht, 2026-02-18

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from scipy.io import loadmat
from scipy.linalg import expm
from IPython.display import Audio, display, HTML

import pyFDN

## Parameters

In [ ]:
fs = 48000
rt = 3.0
ir_len = int(rt * fs)
g = pyFDN.db_to_lin(pyFDN.rt_to_slope(rt, fs))

param_dir = Path("resources", "colorless_FDN")
delay_set = 1
N = 16

## Load parameters from mat file

`load_colorless_params(path, g)` loads m, A, B, C from the mat file, builds Ag = expm(skew(A)) @ diag(g^m) using `pyFDN.skew`, and returns (m_int, Ag, B, C, D) for use with dss2impz.

In [ ]:
def load_colorless_params(path):
    """Load colorless FDN parameters from a .mat file. Returns (m_int, Ag, B, C, D)."""
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"Parameter file not found: {path}")
    data = loadmat(path)
    m = np.asarray(data["m"], dtype=np.float64).ravel()
    A = np.asarray(data["A"], dtype=np.float64)
    B = np.asarray(data["B"], dtype=np.float64).ravel().reshape(-1, 1)
    C = np.asarray(data["C"], dtype=np.float64)
    if C.ndim == 1:
        C = C.reshape(1, -1)
    D = np.zeros((1, 1))
    A_skew = pyFDN.skew(A)
    A = expm(A_skew)
    m_int = np.round(m).astype(np.int64)
    return m_int, A, B, C, D

In [ ]:
path_optim = param_dir / f"param_N{N}_d{delay_set}.mat"
m, A, B, C, D = load_colorless_params(path_optim)
Gamma = np.diag(g ** m)
Ag = A @ Gamma
ir_optim = pyFDN.dss_to_impz(ir_len, m, Ag, B, C, D).squeeze()

## Compare to initialization parameters

In [ ]:
path_init = param_dir / f"param_init_N{N}_d{delay_set}.mat"
m_i, A_i, B_i, C_i, D_i = load_colorless_params(path_init)
Gamma = np.diag(g ** m_i)
Ag_i = A_i @ Gamma
ir_init = pyFDN.dss_to_impz(ir_len, m_i, Ag_i, B_i, C_i, D_i).squeeze()


## Plot impulse responses

In [ ]:
t = np.arange(ir_len) / fs
plt.figure(figsize=(10, 3))
plt.plot(t, pyFDN.mulaw_encode(ir_optim), alpha=0.8, lw=0.6, label="Optimized")
plt.plot(t, pyFDN.mulaw_encode(ir_init), alpha=0.8, lw=0.6, label="Random Initialization")
plt.xlabel("Time [s]")
plt.ylabel("Amplitude [mu-law]")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

display(HTML("<h4>Random Initialization</h4>"))
display(Audio(ir_init, rate=fs))

display(HTML("<h4>Optimized</h4>"))
display(Audio(ir_optim, rate=fs))